# Recommendation System

**Objective:** Build cold-start-safe recommendations using popularity and smoothed ratings.

The code is split into visible, explainable steps for a demonstration video.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'Tourism Dataset'
PROCESSED = ROOT / 'data' / 'processed'
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

In [ ]:
master = pd.read_csv(PROCESSED/'master_dataset.csv')
users, attractions, interactions = master.UserId.nunique(), master.AttractionId.nunique(), len(master)
sparsity = 1 - interactions/(users*attractions)
pd.Series({'users':users, 'attractions':attractions, 'interactions':interactions, 'matrix_sparsity':sparsity})

In [ ]:
global_mean = master.Rating.mean()
recommendations = master.groupby(['AttractionId','Attraction','AttractionType','AttractionCity','AttractionCountry'], dropna=False).agg(interactions=('TransactionId','size'), mean_rating=('Rating','mean')).reset_index()
minimum_interactions = max(3, int(recommendations.interactions.quantile(.5)))
recommendations['recommendation_score'] = recommendations.interactions/(recommendations.interactions+minimum_interactions)*recommendations.mean_rating + minimum_interactions/(recommendations.interactions+minimum_interactions)*global_mean
recommendations['reason'] = 'Popularity and smoothed average rating'
recommendations.sort_values('recommendation_score', ascending=False).head(10)

In [ ]:
def recommend(attraction_type=None, city=None, k=10):
    result = recommendations.copy()
    if attraction_type: result = result[result.AttractionType == attraction_type]
    if city: result = result[result.AttractionCity == city]
    return result.sort_values('recommendation_score', ascending=False).head(k)
recommend(k=5)

In [ ]:
recommendations.to_csv(ROOT/'models'/'recommendation'/'popular_recommendations.csv', index=False)
print('Popularity baseline selected because sparse user-attraction interactions make collaborative filtering less stable.')